In [ ]:
pip install langchain langchain-openai langchain-community duckduckgo-search langgraph langchain-mcp-adapters

In [ ]:
import asyncio
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

class EmailState(TypedDict):
    recipient_email: str
    recipient_name: str
    context: str
    draft_subject: str
    draft_body: str
    feedback: str
    is_approved: bool
    status: str

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")

# 1. Node: Draft the email
def draft_node(state: EmailState):
    if state.get("feedback"):
        prompt = f"""Rewrite this cold email based on the feedback.
        Recipient: {state['recipient_name']}
        Feedback: {state['feedback']}
        Previous Draft: {state.get('draft_body')}

        Format output exactly as:
        SUBJECT: [Your Subject Line]
        BODY: [Your Email Body]"""
    else:
        prompt = f"""Write a cold email to {state['recipient_name']} regarding {state['context']}.
        Format output exactly as:
        SUBJECT: [Your Subject Line]
        BODY: [Your Email Body]"""

    response = llm.invoke(prompt).content

    # Safely parse the subject and body using string splitting
    try:
        parts = response.split("BODY:")
        subject = parts[0].replace("SUBJECT:", "").strip()
        body = parts[1].strip()
        return {"draft_subject": subject, "draft_body": body}
    except Exception:
        return {"draft_subject": "Connecting", "draft_body": response}

# 2. Node: Review the draft
def review_node(state: EmailState):
    body = state.get("draft_body", "")

    # Threshold check: Must be longer than 30 words
    if len(body.split()) > 30:
        return {"is_approved": True, "feedback": "Approved"}
    else:
        return {"is_approved": False, "feedback": "The email is too short. Please add more personalized context."}

def route_email(state: EmailState):
    return "send" if state["is_approved"] else "draft"

# 3. Node: Send via Gmail MCP
async def send_node(state: EmailState):
    print("\n[Connecting to Gmail MCP Server...]")
    # Initialize the stdio-based MCP client
    client = MultiServerMCPClient({
        "gmail": {
            "command": "npx",
            "args": ["-y", "@gptscript-ai/gmail-mcp-server"],
            "transport": "stdio"
        }
    })

    tools = await client.get_tools()

    # Locate the sending tool from the MCP server
    send_tool = next((t for t in tools if "send" in t.name.lower()), None)

    if send_tool:
        print(f"\n[Sending Email to {state['recipient_email']}]")
        await send_tool.ainvoke({
            "to": state["recipient_email"],
            "subject": state["draft_subject"],
            "body": state["draft_body"]
        })
        return {"status": "sent"}
    else:
        print("\n[Error: Could not find send tool on MCP server]")
        return {"status": "failed"}

# 4. Build and run the asynchronous graph
async def run_email_agent():
    workflow = StateGraph(EmailState)
    workflow.add_node("draft", draft_node)
    workflow.add_node("review", review_node)
    workflow.add_node("send", send_node)

    workflow.set_entry_point("draft")
    workflow.add_edge("draft", "review")
    workflow.add_conditional_edges("review", route_email, {"send": "send", "draft": "draft"})
    workflow.add_edge("send", END)

    app = workflow.compile()

    test_input = {
        "recipient_email": "test@example.com", # Replace with your email for testing
        "recipient_name": "Professor Smith",
        "context": "Expressing interest in joining their AI research lab as a graduate student next fall."
    }

    # We use ainvoke because the MCP send node uses async/await
    final_state = await app.ainvoke(test_input)
    print("\n--- Final Status ---")
    print(final_state["status"])

if __name__ == "__main__":
    asyncio.run(run_email_agent())

ModuleNotFoundError: No module named 'langchain_openai'